# 02 — Baseline HHI Analysis (Contractor-Level)

Calculates the **baseline** view of market concentration: how concentrated is federal IT spending across *contractors* (the entities receiving money)?

This is the denominator of our concentration gap analysis. We compare this unconcentrated contractor-level market against the highly concentrated platform-level market (Notebook 04).

Key metrics:
- **HHI** (Herfindahl-Hirschman Index): 0–10,000 scale. <1,500 = unconcentrated, 1,500–2,500 = moderate, >2,500 = highly concentrated
- **C4**: Share of top 4 entities
- **Gini coefficient**: Inequality of spending distribution
- **Old vs New comparison**: Primes-only baseline vs merged baseline

---

In [ ]:
import pandas as pd
import numpy as np
import os, sys, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

baseline_mod = _import_module('baseline_merged_hhi',
    os.path.join(PROJECT_ROOT, 'notebooks', '02_base_analysis', 'baseline_merged_hhi.py'))
merge_mod = _import_module('create_merged_dataset',
    os.path.join(PROJECT_ROOT, 'notebooks', '01_prep', 'create_merged_dataset.py'))

## 1. Load Datasets

In [ ]:
MERGED_PATH   = os.path.join(PROJECT_ROOT, 'data', '02_processed', '02_merged', 'merged_dataset.csv')
FILTERED_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '01_filtered', 'prime_services_filtered.csv')

merged_df = pd.read_csv(MERGED_PATH)
print(f'Merged dataset: {len(merged_df):,} records, ${merged_df["dollars"].sum()/1e9:.1f}B')

# Also load filtered primes for old-vs-new comparison
primes_df = merge_mod.load_primes(FILTERED_PATH)

## 2. Contractor-Level HHI on Merged Dataset

Group all records by `contractor` (resolved Parent UEI), sum dollars, and compute HHI.

In [ ]:
result = baseline_mod.calculate_contractor_hhi(merged_df)

## 3. Market Structure Visualization

In [ ]:
# Top 30 contractors — share of total spending
cs = result['contractor_spending'].head(30).copy()
total = result['total_spending']
cs['share_pct'] = (cs['dollars'] / total * 100).round(2)

print(f'Top 30 contractors account for {cs["share_pct"].sum():.1f}% of spending')
print(f'Remaining {result["n_contractors"] - 30:,} contractors share the other {100 - cs["share_pct"].sum():.1f}%')
print()
for i, (entity, row) in enumerate(cs.iterrows(), 1):
    bar = '#' * int(row['share_pct'] * 4)
    print(f'{i:3d}. {row["name"][:40]:40s} {row["share_pct"]:5.2f}%  {bar}')

In [ ]:
# Lorenz curve data (cumulative share)
all_shares = result['market_shares'].sort_values(ascending=True).values
cum_share = np.cumsum(all_shares)
n = len(all_shares)
x = np.arange(1, n + 1) / n * 100  # cumulative % of firms

print(f'Gini coefficient: {result["gini"]:.4f}')
print(f'Top 1% of contractors hold ~{cum_share[-max(1,int(n*0.01)):].sum() - cum_share[-max(1,int(n*0.01))-1:][0] if n > 100 else all_shares[-1]:.1f}% of spending')
print(f'Bottom 50% of contractors hold ~{cum_share[n//2]:.1f}% of spending')

## 4b. Annual Contractor-Level HHI Trends

How has contractor-level market concentration evolved over the study period? This gives context for the platform-level trends in Notebook 04.

In [ ]:
# Annual contractor-level HHI
print('Annual contractor-level HHI (merged dataset):')
print(f'{"FY":>6s}  {"HHI":>8s}  {"Classification":>28s}  {"Contractors":>12s}  {"C4 (%)":>8s}  {"Spending":>12s}')
print('-' * 82)

for fy in sorted(merged_df['fiscal_year'].dropna().unique()):
    if fy < 2017 or fy > 2024:
        continue
    fy_data = merged_df[merged_df['fiscal_year'] == fy]
    fy_spending = fy_data.groupby('contractor')['dollars'].sum()
    fy_total = fy_spending.sum()
    fy_shares = fy_spending / fy_total * 100
    fy_hhi = (fy_shares ** 2).sum()
    fy_c4 = fy_shares.nlargest(4).sum()
    fy_n = len(fy_spending)
    classification = baseline_mod.classify_hhi(fy_hhi)
    print(f'FY{int(fy):>4d}  {fy_hhi:>8,.1f}  {classification:>28s}  {fy_n:>12,}  {fy_c4:>7.1f}%  ${fy_total/1e9:>9.1f}B')

## 4c. How the Merge Changes the Entity Landscape

The merge introduces new entities: subcontractors that weren't visible in the primes-only view. Who are these new entities, and how do they affect concentration?

In [ ]:
# Entities that appear ONLY through subcontracts (not as prime contractors)
prime_entities = set(primes_df['resolved_entity'].unique())
merged_entities = set(merged_df['contractor'].unique())
sub_only_entities = merged_entities - prime_entities

sub_only_df = merged_df[merged_df['contractor'].isin(sub_only_entities)]
sub_only_spending = sub_only_df.groupby(['contractor', 'contractor_name']).agg(
    dollars=('dollars', 'sum'),
    records=('dollars', 'count')
).sort_values('dollars', ascending=False)

print(f'Entities visible only through subcontracts: {len(sub_only_entities):,}')
print(f'Their total spending: ${sub_only_spending["dollars"].sum()/1e9:.2f}B '
      f'({sub_only_spending["dollars"].sum()/merged_df["dollars"].sum()*100:.1f}% of total)')
print(f'\nTop 20 subcontract-only entities:')
for i, ((entity, name), row) in enumerate(sub_only_spending.head(20).iterrows(), 1):
    print(f'  {i:2d}. {name[:45]:45s} ${row["dollars"]/1e6:>10,.1f}M  ({row["records"]:,} records)')

## 5. Key Takeaway

The contractor-level market is **unconcentrated** by standard antitrust thresholds. Thousands of firms compete for federal IT work. This is the baseline against which we measure platform concentration — if spending is ultimately flowing through just a few cloud platforms, the *effective* concentration is far higher than the contractor view suggests.

---

**Next:** [03a — Cloud Classification](../03_multi-stage_attribution_pipeline/03a_cloud_classification.ipynb)